# tspgnn: multi-seed GPU training and evaluation

This notebook trains the distance-only GNN (`tspgnn/`) with several random seeds on a GPU,
then evaluates every seed with the repository's own benchmark harness (`tspbench`),
both greedy + 2-opt and the GNN-guided search.

**To run:** `Runtime → Change runtime type → A100 GPU` (with High-RAM), then `Runtime → Run all`.
The defaults are sized for an A100 runtime, which has 12 CPU cores and 80+ GB of RAM. On a T4, use 3 seeds, 40000 instances and turn off `PARALLEL_SEEDS`.
Allow Google Drive access when asked.

Everything (training data, checkpoints, logs, benchmark results) is saved to
`MyDrive/tsp_gnn_solver/`. If the Colab session disconnects, just `Run all` again:
finished steps are skipped and training resumes from the last finished epoch.

The final table is written to `MyDrive/tsp_gnn_solver/<run>/SEEDS.md`.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown Git branch to run. Use `main` once the open PRs are merged.
BRANCH = "claude/colab-gpu-training-xtmix1"  #@param {type:"string"}
#@markdown Model seeds to train, comma-separated.
SEEDS = "0,1,2,3,4"  #@param {type:"string"}
#@markdown Training set (LKH-3 labelled, n = 20 to 100, mixed distance types). The shipped CPU checkpoint used 40000.
TRAIN_NUM = 100000  #@param {type:"integer"}
EPOCHS = 20  #@param {type:"integer"}
HIDDEN = 64  #@param {type:"integer"}
LAYERS = 12  #@param {type:"integer"}
#@markdown Train all seeds at once on the one GPU (the model is small, so one run leaves the A100 mostly idle). Each run holds its own copy of the data in RAM.
PARALLEL_SEEDS = True  #@param {type:"boolean"}
#@markdown Test instances per benchmark suite (Fu TSP500/1000 have 128, TSP10000 has 16).
EVAL_LIMIT = 128  #@param {type:"integer"}
#@markdown Guided-search budget in seconds per city (0.002 is what results/SEARCH.md uses).
TIME_PER_NODE = 0.002  #@param {type:"number"}
#@markdown Also evaluate the checkpoint shipped in the repo (trained on CPU) for comparison.
EVAL_SHIPPED_CHECKPOINT = True  #@param {type:"boolean"}
#@markdown Save to Google Drive, so a disconnected session can resume.
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown Smoke test: a tiny model and data, about 5 minutes end to end even on CPU.
TINY = False  #@param {type:"boolean"}

In [ ]:
import os, subprocess, sys, json, glob, shutil

if TINY:
    TRAIN_NUM, EPOCHS, HIDDEN, LAYERS, EVAL_LIMIT, TIME_PER_NODE = 200, 2, 16, 2, 2, 0.0005
    SEEDS = "0,1"
    PARALLEL_SEEDS = True
SEED_LIST = [int(s) for s in SEEDS.split(",") if s.strip()]

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/tsp_gnn_solver"
else:
    WORK = os.path.abspath("tsp_gnn_solver_work")
if TINY:
    WORK += "_tiny"
REPO = os.path.abspath("tsp_gnn_solver")
DATA = f"{WORK}/data/n{TRAIN_NUM}"
RUN = f"{WORK}/runs/h{HIDDEN}_l{LAYERS}_e{EPOCHS}_n{TRAIN_NUM}"
BENCH = f"{WORK}/bench_data"
EVAL = f"{RUN}/eval_t{TIME_PER_NODE:g}_lim{EVAL_LIMIT}"
os.makedirs(WORK, exist_ok=True)


def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")


print("work dir:", WORK)

## 1. Code and dependencies

In [ ]:
if os.path.isdir(f"{REPO}/.git"):
    sh(f"git fetch -q origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD")
else:
    sh(f"git clone -q -b {BRANCH} https://github.com/pmandros/tsp_gnn_solver.git {REPO}", cwd=".")
sh("git log --oneline -1")
# Colab ships torch, numpy, scipy and numba. elkai is LKH-3 (labels and references).
sh(f"{sys.executable} -m pip install -q elkai==2.0.1 && {sys.executable} -m pip install -q -e . --no-deps")
sys.path.insert(0, REPO)

import torch
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none, training runs on CPU (slow)")
print("CPU cores:", os.cpu_count())

## 2. Training data

Generated once and kept in Drive. LKH-3 labelling runs on the CPU
(about 15 minutes for 100,000 instances on an A100 runtime's 12 cores).

In [ ]:
if not os.path.exists(f"{DATA}/DONE"):
    os.makedirs(DATA, exist_ok=True)
    local = "/tmp/tspgnn_data"  # write locally, then copy: Drive is slow for many small writes
    shutil.rmtree(local, ignore_errors=True)
    sh(f"python scripts/make_data.py --out {local}/train.pt --num {TRAIN_NUM} --seed 1")
    sh(f"python scripts/make_data.py --out {local}/val.pt --num {max(20, min(600, TRAIN_NUM // 60))} "
       f"--seed 2 --store-d")
    for f in glob.glob(f"{local}/*.pt"):
        shutil.copy(f, DATA)
    open(f"{DATA}/DONE", "w").close()
print(sorted(os.listdir(DATA)))

## 3. Train one model per seed

Each seed is checkpointed every epoch. Re-running this cell resumes unfinished seeds
and skips finished ones. `best.pt` is the epoch with the lowest validation gap.

In [ ]:
def finished(run_dir):
    try:
        return len(json.load(open(f"{run_dir}/log.json"))) >= EPOCHS
    except (OSError, ValueError):
        return False


def train_cmd(seed, threads):
    return (f"python scripts/train.py --train '{DATA}/train*.pt' --val {DATA}/val.pt "
            f"--out {RUN}/seed{seed} --epochs {EPOCHS} --hidden {HIDDEN} --layers {LAYERS} "
            f"--seed {seed} --threads {threads} --resume")


todo = [s for s in SEED_LIST if not finished(f"{RUN}/seed{s}")]
print("already trained:", [s for s in SEED_LIST if s not in todo])
if todo and PARALLEL_SEEDS:
    # One process per seed on the same GPU; each writes its own log, printed as it progresses.
    import time
    threads = max(1, os.cpu_count() // len(todo))
    procs = {}
    for seed in todo:
        os.makedirs(f"{RUN}/seed{seed}", exist_ok=True)
        out = open(f"{RUN}/seed{seed}/train.out", "a")
        procs[seed] = subprocess.Popen(train_cmd(seed, threads), shell=True, cwd=REPO,
                                       stdout=out, stderr=subprocess.STDOUT)
    seen = {s: 0 for s in todo}
    while True:
        done = all(p.poll() is not None for p in procs.values())
        for seed in todo:  # echo each seed's per-epoch summaries
            lines = open(f"{RUN}/seed{seed}/train.out").read().splitlines()
            for line in lines[seen[seed]:]:
                if line.startswith("{") or "resumed" in line or "Error" in line:
                    print(f"seed {seed}: {line}", flush=True)
            seen[seed] = len(lines)
        if done:
            break
        time.sleep(30 if not TINY else 1)
    failed = [s for s, p in procs.items() if p.returncode]
    if failed:
        raise RuntimeError(f"seeds {failed} failed, see {RUN}/seed<k>/train.out")
else:
    for seed in todo:
        sh(train_cmd(seed, os.cpu_count()))

for seed in SEED_LIST:
    log = json.load(open(f"{RUN}/seed{seed}/log.json"))
    best = min(log, key=lambda r: r["val_greedy_gap"])
    print(f"seed {seed}: best val greedy gap {100 * best['val_greedy_gap']:.2f}% at epoch {best['epoch']}, "
          f"{log[-1]['time_s'] / 60:.0f} min")

## 4. Benchmark data and references

Fu et al. TSP500, TSP1000 and TSP10000, Manhattan n = 500
(a distance type held out of training) and random non-metric n = 1000.
Reference tour lengths (LKH-3 / Concorde) for all of them ship in `data/refs/`.

In [ ]:
SUITES = ["tsp500", "tsp1000", "manhattan500:num=128", "nonmetric1000:num=128"]
BIG_SUITES = ["tsp10000"]  # run with fewer workers: each n = 10,000 matrix is 800 MB
if TINY:
    SUITES, BIG_SUITES = ["tsp500", "nonmetric100:num=128"], []

if not os.path.exists(f"{BENCH}/fu"):
    sh(f"python scripts/download_benchmarks.py --data-dir {BENCH} --no-tsplib")
os.makedirs(f"{BENCH}/refs", exist_ok=True)
for f in glob.glob(f"{REPO}/data/refs/*.json"):
    if not os.path.exists(f"{BENCH}/refs/{os.path.basename(f)}"):
        shutil.copy(f, f"{BENCH}/refs")
for suite in SUITES + BIG_SUITES:
    # References for all these suites ship in data/refs; compute any that are missing with LKH-3.
    if not os.path.exists(f"{BENCH}/refs/{suite.replace(':', '_')}.json"):
        sh(f"python -m tspbench reference --suite {suite} --limit {EVAL_LIMIT} --solver lkh "
           f"--workers {os.cpu_count()} --data-dir {BENCH}")

## 5. Evaluate every seed

For each model: greedy + 2-opt, and the GNN-guided search with the same time budget per city.
The distance-guided search (the same search with edges ranked by distance, no model) runs once
as the baseline, on this machine, so the time budgets match.

In [ ]:
ENV = {"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "NUMBA_NUM_THREADS": "1"}
COMMON = f"--limit {EVAL_LIMIT} --seeds 0 --data-dir {BENCH}"
RUNS = [(EVAL, SUITES, os.cpu_count())]
if BIG_SUITES:
    RUNS.append((f"{EVAL}_big", BIG_SUITES, min(4, os.cpu_count())))


def gnn_solvers(ckpt):
    return (f'--solver "callable:fn=tspgnn.api:solve,checkpoint={ckpt},name=gnn+2opt" '
            f'--solver "callable:fn=tspgnn.api:solve_search,guide=gnn,checkpoint={ckpt},'
            f'time_per_node={TIME_PER_NODE},name=gnn+search,stochastic=true"')


jobs = {f"seed{s}": gnn_solvers(f"{RUN}/seed{s}/best.pt") for s in SEED_LIST}
jobs["baseline"] = (f'--solver "callable:fn=tspgnn.api:solve_distance_greedy,name=dist+2opt" '
                    f'--solver "callable:fn=tspgnn.api:solve_search,guide=dist,'
                    f'time_per_node={TIME_PER_NODE},name=dist+search,stochastic=true"')
if EVAL_SHIPPED_CHECKPOINT:
    jobs["shipped_cpu"] = gnn_solvers(f"{REPO}/checkpoints/tspgnn.pt")

for root, suites, workers in RUNS:
    suite_args = " ".join(f"--suite {s}" for s in suites)
    for name, solvers in jobs.items():
        out = f"{root}/{name}"
        if os.path.exists(f"{out}/summary.csv"):
            print(f"{name} on {suites}: already evaluated")
            continue
        sh(f"python -m tspbench run {suite_args} {solvers} {COMMON} --workers {workers} --out {out}",
           env=ENV)

## 6. Results across seeds

In [ ]:
dirs = " ".join(f"--eval-dir {root}" for root, _, _ in RUNS)
sh(f"python scripts/seed_summary.py {dirs} --out {RUN}/SEEDS.md")
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN}/SEEDS.md").read()))
print("saved to", f"{RUN}/SEEDS.md")

Checkpoints are in `runs/<run>/seed*/best.pt` in the work folder. To use one elsewhere:
`tspgnn.api.solve(distance_matrix, checkpoint="…/best.pt")`.